# Facebook Political Advertising Analytics at Scale

This notebook analyzes the **Facebook Ad Library dataset for Australian political advertisements** using **Apache Spark**.

### Objectives

- Process large-scale political advertisement data
- Perform keyword frequency analysis
- Analyze advertisement spending
- Identify top political funders
- Investigate COVID-related political campaigns
- Evaluate advertising efficiency using CPM (Cost per 1000 impressions)

### Technologies

- PySpark
- Pandas
- Matplotlib
- HDFS

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, array, explode, split, lower, regexp_replace, count, sum
)

import matplotlib.pyplot as plt
import pandas as pd

## Start Spark Session

In [ ]:
spark = SparkSession.builder \
    .appName("Facebook Political Ads Analysis") \
    .getOrCreate()

## Load Dataset from HDFS

In [ ]:
datafile = spark.read.option(
    "multiLine", "false"
).json("hdfs:///data/ProjectDatasetFacebookAU/adsAU/*")

datafile.cache()

datafile.printSchema()

## Initial Data Cleaning

In [ ]:
ads = datafile.filter(
    (col("funding_entity").isNotNull()) &
    (col("page_name").isNotNull()) &
    ((col("ad_creative_bodies").isNotNull()) |
     (col("ad_creative_body").isNotNull()))
)

ads = ads.dropDuplicates(["id"])

## Flatten Advertisement Text

In [ ]:
ads = ads.withColumn(
    "text_array",
    when(
        col("ad_creative_bodies").isNotNull(),
        col("ad_creative_bodies")
    ).otherwise(array(col("ad_creative_body")))
)

## Extract Ad Text and Metrics

In [ ]:
ads_flat = ads.select(
    "ad_creation_time",
    explode("text_array").alias("ad_text"),
    "page_name",
    "funding_entity",
    col("spend.upper_bound").cast("int").alias("spend_upper"),
    col("impressions.upper_bound").cast("int").alias("impressions_upper")
)

ads_clean = ads_flat.filter(
    (col("ad_text") != "") &
    col("ad_text").isNotNull()
)

## Tokenize Advertisement Text

In [ ]:
ads_split = ads_clean.withColumn(
    "word",
    explode(
        split(
            lower(col("ad_text")),
            "\\W+"
        )
    )
)

## Stopword List

In [ ]:
stopwords = [
"a","about","above","after","again","against","all","am","an","and",
"any","are","aren't","as","at","be","because","been","before","being",
"below","between","both","but","by","can","cannot","could","couldn't",
"did","didn't","do","does","doesn't","doing","don't","down","during",
"each","few","for","from","further","had","hadn't","has","hasn't",
"have","haven't","having","he","he'd","he'll","he's","her","here",
"here's","hers","herself","him","himself","his","how","how's","i",
"i'd","i'll","i'm","i've","if","in","into","is","isn't","it","it's",
"its","itself","let's","me","more","most","mustn't","my","myself",
"no","nor","not","of","off","on","once","only","or","other","ought",
"our","ours","ourselves","out","over","own","same","shan't","she",
"she'd","she'll","she's","should","shouldn't","so","some","such",
"than","that","that's","the","their","theirs","them","themselves",
"then","there","there's","these","they","they'd","they'll","they're",
"they've","this","those","through","to","too","under","until","up",
"very","was","wasn't","we","we'd","we'll","we're","we've","were",
"weren't","what","what's","when","when's","where","where's","which",
"while","who","who's","whom","why","why's","with","won't","would",
"wouldn't","you","you'd","you'll","you're","you've","your","yours",
"yourself","yourselves","s","d","t","m","re","ll","ve"
]

## Word Filtering

In [ ]:
word_filter = ads_split.filter(
    (col("word") != "") &
    (~col("word").isin(stopwords))
)

## Keyword Frequency Analysis

In [ ]:
word_count = word_filter.groupBy("word") \
    .count() \
    .orderBy("count", ascending=False)

word_count.show(20)

## Prepare Spend Data

In [ ]:
ads_split_clean = ads_split.drop("spend_upper")

spend_data = ads_clean.select(
    "ad_text",
    col("spend_upper").alias("spend_amt")
)

## Join Word Data with Spend Information

In [ ]:
join_spend = ads_split_clean.join(
    spend_data,
    on="ad_text",
    how="left"
)

## Keyword Spending Analysis

In [ ]:
join_spend_clean = join_spend.filter(
    (col("word") != "") &
    (~col("word").isin(stopwords)) &
    (col("spend_amt").isNotNull())
)

word_spend = join_spend_clean.groupBy("word") \
    .sum("spend_amt") \
    .withColumnRenamed("sum(spend_amt)", "total_spend") \
    .orderBy("total_spend", ascending=False)

word_spend.show(20)

## Top Funding Entities 

In [ ]:
investors = ads_clean.groupBy("funding_entity") \
    .sum("spend_upper") \
    .withColumnRenamed("sum(spend_upper)", "total_spend") \
    .orderBy("total_spend", ascending=False)

investors.show(20)

## Top Facebook Pages

In [ ]:
top_pages = ads_clean.groupBy("page_name") \
    .sum("spend_upper") \
    .withColumnRenamed("sum(spend_upper)", "total_spend") \
    .orderBy("total_spend", ascending=False)

top_pages.show(20)

## Advertisement Volume Over Time

In [ ]:
ad_vol = ads_clean.groupBy("ad_creation_time") \
    .agg(count("*").alias("ad_count")) \
    .orderBy("ad_creation_time")

ad_vol.show()

## COVID Campaign Filtering

In [ ]:
ads_covid = ads_clean.filter(
    lower(col("ad_text")).contains("covid")
)

ads_covid.select(
    "ad_creation_time",
    "funding_entity",
    "page_name",
    "ad_text",
    "spend_upper"
).show(5, truncate=False)

## COVID Spending Analysis

In [ ]:
spend_covid = ads_covid.agg(
    sum("spend_upper").alias("covid_spend")
)

spend_covid.show()

## CPM Calculation

In [ ]:
reach_df = ads_clean.select(
    "ad_text",
    "funding_entity",
    "page_name",
    col("spend_upper").cast("double").alias("spend"),
    col("impressions_upper").cast("double").alias("impressions")
).filter(
    col("spend").isNotNull() &
    col("impressions").isNotNull() &
    (col("impressions") > 0)
)

reach_df = reach_df.withColumn(
    "cpm",
    (col("spend") / col("impressions")) * 1000
)

## High CPM and Low CPM ADS

In [ ]:
high_cpm_ads = reach_df.filter(
    col("cpm") > 100
).orderBy(col("cpm").desc())

high_cpm_ads.select(
    "funding_entity",
    "page_name",
    "spend",
    "impressions",
    "cpm"
).show(10, truncate=False)

low_cpm_ads = reach_df.filter(
    col("cpm") < 10
).orderBy(col("cpm"))

low_cpm_ads.select(
    "funding_entity",
    "page_name",
    "spend",
    "impressions",
    "cpm"
).show(10, truncate=False)

## CPM Distribution Visualization

In [ ]:
reach_sample = reach_df.select("cpm").dropna().toPandas()

plt.figure(figsize=(10,5))

plt.hist(
    reach_sample["cpm"],
    bins=100,
    color="red",
    edgecolor="black"
)

plt.xlabel("CPM (Cost per 1000 Impressions)")
plt.ylabel("Number of Ads")
plt.title("Distribution of CPM Across All Ads")

plt.tight_layout()

plt.show()